In [ ]:
## IMPORTS & SETUP


import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors
import light_curve as lc

import warnings
# Disabling FutureWarnings:
warnings.filterwarnings('ignore', category=FutureWarning)

# Feature names (without linear features):
feature_names = ['mean', 'weightedMean', 'std', 'median', 'amplitude', 'beyond1Std', 'cusum', 'IPR10',
                 'kurtosis', 'MPR40_5', 'MPR20_10', 'maxSlope', 'medianAbsDev', 'medianBRP10',
                 'percentAmplitude', 'meanVariance', 'andersonDarlingNorm', 'chi2', 'skew', 'stetsonK']

In [ ]:
## LOAD DATA


# Ids and statistical features already computed from light curves in g and r bands of known magnetic cataclysmic variable stars (positive objects):
positive_Ids = pd.read_csv('data/magnetic_cvs_objectId.csv').values.flatten()
positive_g = pd.read_parquet('../../data/magcvs/features_g.parquet')
positive_r = pd.read_parquet('../../data/magcvs/features_r.parquet')

# Light curves of all objects scanned last night:
other_lc = pd.read_parquet('../../data/All_2020').iloc[:2000]
# The format of the data should be as follows:
# 'objectId': str                   The identifier of the object,
# 'i:jd': Iterable[float]           The Julian date for every point in the light curve,
# 'i:magpsf': Iterable[float]       The magnitude of the object for every point in its light curve,
# 'i:sigmapsf': Iterable[float]     The error of the magnitude for every point in its light curve,
# 'i:fid': Iterable[int]            For every point in the light curve, wether it is from the g or r band (1 -> g-band, 2 -> r-band).


In [3]:
## DATA PREPARATION & FEATURE EXTRACTION


# Splitting the data from last night by filter and removing potential already known positive objects:
def sort_negative(negative_lc, positive_Ids):
    """
    Function specific to the format of the data like the one in ../../data/All_2020/  
    Removes positive objects and returns the light curves by filter.
    """

    # Removing potential positive class objects from the negative class:
    negative_Ids = negative_lc['objectId'].values
    intersect = np.intersect1d(negative_Ids, positive_Ids)
    negative_lc = negative_lc[~np.isin(negative_Ids, intersect)]
    
    # Splitting data by filter:
    negative_lc_g = pd.DataFrame(columns=['objectId','time_range (yr)', 'nb_of_points', 'i:jd', 'i:magpsf', 'i:sigmapsf'])
    negative_lc_r = pd.DataFrame(columns=['objectId','time_range (yr)', 'nb_of_points', 'i:jd', 'i:magpsf', 'i:sigmapsf'])
    for _, row in negative_lc.iterrows():
        jd_g, magpsf_g, sigmapsf_g = np.array([]), np.array([]), np.array([])
        jd_r, magpsf_r, sigmapsf_r = np.array([]), np.array([]), np.array([])
        for index, fid in enumerate(row['i:fid']):
            if fid == 1:
                jd_g = np.append(jd_g, row['i:jd'][index])
                magpsf_g = np.append(magpsf_g, row['i:magpsf'][index])
                sigmapsf_g = np.append(sigmapsf_g, row['i:sigmapsf'][index])
            else:
                jd_r = np.append(jd_r, row['i:jd'][index])
                magpsf_r = np.append(magpsf_r, row['i:magpsf'][index])
                sigmapsf_r = np.append(sigmapsf_r, row['i:sigmapsf'][index])
        if len(jd_g) >= 4:
            new_row_g = pd.DataFrame([dict(zip(negative_lc_g.columns, [row['objectId'], round((max(jd_g)-min(jd_g))/365, 2), len(jd_g), jd_g, magpsf_g, sigmapsf_g]))])
            negative_lc_g = pd.concat([negative_lc_g, new_row_g], ignore_index=True)
        if len(jd_r) >= 4:
            new_row_r = pd.DataFrame([dict(zip(negative_lc_r.columns, [row['objectId'], round((max(jd_r)-min(jd_r))/365, 2), len(jd_r), jd_r, magpsf_r, sigmapsf_r]))])
            negative_lc_r = pd.concat([negative_lc_r, new_row_r], ignore_index=True)

    return negative_lc_g, negative_lc_r
other_lc_g, other_lc_r = sort_negative(other_lc, positive_Ids)

# Extracting the features from the light curves:
def extract_features(light_curve_data: pd.DataFrame) -> pd.DataFrame:
    """Extracts statistical features from light curve data using the light_curve library.

    Args:
        light_curve_data (pd.DataFrame): A DataFrame containing light curve data with columns 'i:jd', 'i:magpsf', and 'i:sigmapsf'. Each row should represent the light curve data for a single object.

    Returns:
        pd.DataFrame: A DataFrame containing the extracted features.
    """

    # Initializing features with the light_curve library:
    mean = lc.Mean()
    weighted_mean = lc.WeightedMean()
    standard_deviation = lc.StandardDeviation()
    median = lc.Median()
    amplitude = lc.Amplitude()
    beyond_1_std = lc.BeyondNStd(nstd=1)
    cusum = lc.Cusum()
    inter_percentile_range_10 = lc.InterPercentileRange()
    kurtosis = lc.Kurtosis()
    linear_trend = lc.LinearTrend()
    linear_fit_slope = lc.LinearFit()
    magnitude_percentage_ratio_40_5 = lc.MagnitudePercentageRatio(quantile_numerator=.4, quantile_denominator=.05)
    magnitude_percentage_ratio_20_10 = lc.MagnitudePercentageRatio(quantile_numerator=.2, quantile_denominator=.1)
    maximum_slope = lc.MaximumSlope()
    median_absolute_deviation = lc.MedianAbsoluteDeviation()
    median_buffer_range_percentage_10 = lc.MedianBufferRangePercentage(quantile=.1)
    percent_amplitude = lc.PercentAmplitude()
    mean_variance = lc.MeanVariance()
    anderson_darling_normal = lc.AndersonDarlingNormal()
    chi2 = lc.ReducedChi2()
    skew = lc.Skew()
    stetson_K = lc.StetsonK()

    extractor = lc.Extractor(mean, weighted_mean, standard_deviation, median, amplitude, beyond_1_std,
                            cusum, inter_percentile_range_10, kurtosis, linear_trend, linear_fit_slope,
                            magnitude_percentage_ratio_40_5, magnitude_percentage_ratio_20_10, maximum_slope,
                            median_absolute_deviation, median_buffer_range_percentage_10, percent_amplitude,
                            mean_variance, anderson_darling_normal, chi2, skew, stetson_K)

    # Feature names for the columns of the output DataFrame:
    feature_names = ['mean', 'weightedMean', 'std', 'median', 'amplitude', 'beyond1Std',
                    'cusum', 'IPR10', 'kurtosis', 'linT', 'linT_sigma', 'linT_noise',
                    'linF_slope', 'linF_slope_sigma', 'linF_chi2', 'MPR40_5', 'MPR20_10',
                    'maxSlope', 'medianAbsDev', 'medianBRP10', 'percentAmplitude',
                    'meanVariance', 'andersonDarlingNorm', 'chi2', 'skew', 'stetsonK']

    # Extracting the features:
    features = []
    for line in range(len(light_curve_data)):
        features.append(extractor(light_curve_data['i:jd'][line], light_curve_data['i:magpsf'][line], light_curve_data['i:sigmapsf'][line], sorted=True, check=False))
    light_curve_data[feature_names] = np.vstack(features)

    return light_curve_data.drop(columns=['i:jd', 'i:magpsf', 'i:sigmapsf'])
other_features_g = extract_features(other_lc_g)
other_features_r = extract_features(other_lc_r)

# Normalizing features:
# g-band:
all_g = pd.concat([positive_g, other_features_g]) # Grouping all objects together so that they are then normalized the same way.
all_g = normalize(all_g[feature_names], axis=0)
positive_g[feature_names] = all_g[:len(positive_g)]
other_features_g[feature_names] = all_g[len(positive_g):]
# r-band:
all_r = pd.concat([positive_r, other_features_r])
all_r = normalize(all_r[feature_names], axis=0)
positive_r[feature_names] = all_r[:len(positive_r)]
other_features_r[feature_names] = all_r[len(positive_r):]

In [4]:
## MAIN


def find_candidates(positive: pd.DataFrame, feature_space: pd.DataFrame, n_neighbors: int = 3, candidate_threshold: int = 2, max_candidates: int = 10, feature_names: list[str] | None = None):
    """
    Evaluates candidates for the positive class among given objects in the feature space using the nearest neighbors algorithm on given positive class objects.  
    The candidates are objects that appear more than 'candidate_threshold' times in the nearest neighbors of the positive objects.  
    The inputted DataFrames (positive & feature_space) should contain the same features (feature_names) and have columns 'objectId', 'time_range (yr)', 'nb_of_points', 'class'.  
    Returns the candidates in a DataFrame

    Parameters
    -------
        positive: pd.DataFrame
            DataFrame containing the features of positive class objects to find candidates for.
        feature_space: pd.DataFrame
            DataFrame containing the features of all objects to evaluate.
        n_neighbors: int, default=3
            Number of neighbors for the nearest neighbors algorithm.
        candidate_threshold: int, default=2
            Parameter for candidate evaluation.
        max_candidates: int, default=10
            Maximum number of candidates to return.
        feature_names: list[str] | None, default=None
            List of feature names to use for the nearest neighbors algorithm. If None, default feature names are used.

    Returns
    -------
        candidates: pd.DataFrame
            Candidates for the positive class ordered by the number of times they appear in the nearest neighbors of the positive objects.
    """

    if feature_names is None: # If no feature names are provided, use the default ones (without linear features):
        feature_names = ['mean', 'weightedMean', 'std', 'median', 'amplitude', 'beyond1Std', 'cusum', 'IPR10',
                        'kurtosis', 'MPR40_5', 'MPR20_10', 'maxSlope', 'medianAbsDev', 'medianBRP10',
                        'percentAmplitude', 'meanVariance', 'andersonDarlingNorm', 'chi2', 'skew', 'stetsonK']

    # Finding the nearest neighbors of positive objects:
    neigh = NearestNeighbors(n_neighbors=n_neighbors).fit(feature_space[feature_names])
    neighbors_indices = neigh.kneighbors(positive[feature_names], return_distance=False)
    neighbors = feature_space.iloc[neighbors_indices.flatten()]

    # Ids of the neighbors and the number of times each id appears:
    ids, counts = np.unique(neighbors['objectId'], return_counts=True)

    # Creating the DataFrame that will store the candidates:
    col = ['objectId', 'time_range (yr)', 'nb_of_points']
    candidates = pd.DataFrame(columns=[*col, 'count'])

    # Adding the candidates to the DataFrame:
    for id, count in zip(ids, counts):
        if count > candidate_threshold: # An object is considered as a candidate if it appears more than 'candidate_threshold' times in the neighbors
            candidate = neighbors[neighbors['objectId'] == id]
            new_row = pd.DataFrame([dict(zip(candidates.columns, [candidate['objectId'].values[0], candidate['time_range (yr)'].values[0], candidate['nb_of_points'].values[0], count]))])
            candidates = pd.concat([candidates, new_row], ignore_index=True)

    # Sorting the candidates by the number of times they appear in the neighbors:
    candidates = candidates.sort_values(by='count', ascending=False).reset_index(drop=True)

    return candidates.iloc[:max_candidates] # Returning only the first 'max_candidates' candidates.
candidates_g = find_candidates(positive_g, other_features_g)
candidates_r = find_candidates(positive_r, other_features_r)

In [9]:
print(candidates_g)

       objectId  time_range (yr) nb_of_points count
0  ZTF18abmmazn             0.71           73     6
1  ZTF18abntzed             0.63           49     6
2  ZTF18abdlyeg             0.83          108     5
3  ZTF18abjmftp             0.98          107     5
4  ZTF17aabulaf             0.88           64     4
5  ZTF17aabooqh             0.66           55     3
6  ZTF18aaslfng             0.62           71     3
7  ZTF18aaxcron             0.83          162     3
8  ZTF18abaegid             0.98           98     3
9  ZTF18abbobro             0.99          105     3


In [10]:
print(candidates_r)

       objectId  time_range (yr) nb_of_points count
0  ZTF17aabqhzy             0.96           66     5
1  ZTF18abcvfda             0.91          167     5
2  ZTF18aaioqrj             0.82           82     3
3  ZTF18aaiykoz             0.24           66     3
4  ZTF18abbobro             0.99           97     3
5  ZTF18abmarba             0.96           92     3
6  ZTF18abmsour             0.97           92     3
7  ZTF18abmvfmx             0.65           73     3
8  ZTF18abyxxpf             0.98           67     3
